# AI 3 — Modelling: login-risk pipeline

Compares **rule-based baseline**, **Logistic Regression**, **Random Forest** and **XGBoost** (if installed) on `data/mvp_synthetic_logins.csv`.

**Guard rails**
- The notebook **stops** if the real CSV is missing. There is no mock/placeholder data anywhere in it, so no metrics can be produced before the file arrives.
- Identity/session columns, the target label and `attack_type` are never model inputs.
- Split is **both time-based and user-based** (held-out users, later period only), so no user and no future information leaks into training.
- Risk score is 0–100 for every model. SHAP is used for explanation only.

**Before first run:** check the column names in the *Config* cell against the shared schema (they are assumptions until the schema/CSV is confirmed).

**Rerun:** `jupyter nbconvert --to notebook --execute --inplace notebooks/model_pipeline.ipynb`

In [ ]:
import warnings, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_score,
                             recall_score, f1_score, confusion_matrix,
                             ConfusionMatrixDisplay, precision_recall_curve, roc_curve)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

# ---------------- CONFIG (edit to match the shared schema) ----------------
CSV_PATH   = Path("data/mvp_synthetic_logins.csv")
TARGET     = "label"            # binary target: 1 = malicious/attack login
TIME_COL   = "timestamp"        # event time (parsed to datetime)
USER_COL   = "user_id"          # used ONLY for the user-based split
ID_COLS    = ["session_id", "ip_address", "device_id", "login_id"]  # identity/session fields -> never features
LEAK_COLS  = ["attack_type"]    # label-derived -> never features

TIME_CUTOFF_Q   = 0.70   # test = events after this time quantile
TEST_USER_FRAC  = 0.20   # fraction of users held out entirely for test
VAL_TIME_Q      = 0.80   # inside train: last 20% of time is validation (threshold selection)
FAR_TARGET      = 0.05   # max false-alarm rate allowed when picking the alert threshold
OUT_DIR = Path("outputs"); MODEL_DIR = Path("models")
OUT_DIR.mkdir(exist_ok=True); MODEL_DIR.mkdir(exist_ok=True)

## 0. Load data, train/test split, normalisation

In [ ]:
# --- Load: fails loudly if the real file is not there ---
if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"{CSV_PATH} not found. Get the real file from AI 2 first. "
        "This notebook intentionally never fabricates or mocks data."
    )
df = pd.read_csv(CSV_PATH)
print("shape:", df.shape)
print("columns:", list(df.columns))
display(df.head())

missing = [c for c in (TARGET, TIME_COL, USER_COL) if c not in df.columns]
assert not missing, f"Config columns not in CSV: {missing} -> fix the CONFIG cell"
assert set(df[TARGET].dropna().unique()) <= {0, 1}, "TARGET must be binary 0/1"

df[TIME_COL] = pd.to_datetime(df[TIME_COL])
df = df.sort_values(TIME_COL).reset_index(drop=True)
print("\nclass balance:\n", df[TARGET].value_counts(normalize=True))

In [ ]:
# --- Feature selection (no IDs, no target, no attack_type) ---
df["hour"] = df[TIME_COL].dt.hour
df["dayofweek"] = df[TIME_COL].dt.dayofweek

drop_cols = [c for c in [TARGET, TIME_COL, USER_COL, *ID_COLS, *LEAK_COLS] if c in df.columns]
X_all = df.drop(columns=drop_cols)

# auto-flag ID-like text columns (almost unique per row) and drop them too
id_like = [c for c in X_all.select_dtypes(exclude="number").columns
           if X_all[c].nunique() / len(X_all) > 0.5]
if id_like:
    print("Dropping ID-like columns:", id_like)
    X_all = X_all.drop(columns=id_like)

bool_cols = X_all.select_dtypes("bool").columns
X_all[bool_cols] = X_all[bool_cols].astype(int)
y_all = df[TARGET].astype(int)

NUM_COLS = X_all.select_dtypes("number").columns.tolist()
CAT_COLS = X_all.select_dtypes(exclude="number").columns.tolist()
print("numeric features:", NUM_COLS)
print("categorical features:", CAT_COLS)

In [ ]:
# --- Leak-free split: held-out USERS  x  LATER time period ---
rng = np.random.default_rng(RANDOM_STATE)
users = df[USER_COL].unique()
test_users = set(rng.choice(users, size=max(1, int(len(users) * TEST_USER_FRAC)), replace=False))

t_cut = df[TIME_COL].quantile(TIME_CUTOFF_Q)
is_test_user = df[USER_COL].isin(test_users)

train_mask = (~is_test_user) & (df[TIME_COL] <  t_cut)
test_mask  = ( is_test_user) & (df[TIME_COL] >= t_cut)

# validation = most recent slice of the training period (threshold selection only)
t_val = df.loc[train_mask, TIME_COL].quantile(VAL_TIME_Q)
fit_mask = train_mask & (df[TIME_COL] <  t_val)
val_mask = train_mask & (df[TIME_COL] >= t_val)

X_fit, y_fit = X_all[fit_mask], y_all[fit_mask]
X_val, y_val = X_all[val_mask], y_all[val_mask]
X_test, y_test = X_all[test_mask], y_all[test_mask]

# sanity checks
assert set(df.loc[train_mask, USER_COL]).isdisjoint(set(df.loc[test_mask, USER_COL])), "user leakage"
assert df.loc[train_mask, TIME_COL].max() < df.loc[test_mask, TIME_COL].min(), "time leakage"
for n, y in [("fit", y_fit), ("val", y_val), ("test", y_test)]:
    assert y.nunique() == 2, f"{n} split has only one class -> adjust cutoffs"
    print(f"{n:5s} rows={len(y):>7,}  attack rate={y.mean():.3%}")

In [ ]:
# --- Normalisation / encoding (fit on TRAIN only, via pipelines) ---
def make_preprocessor(scale: bool):
    num_steps = [("imp", SimpleImputer(strategy="median"))]
    if scale:                                   # scaling only matters for Logistic Regression
        num_steps.append(("scale", StandardScaler()))
    transformers = [("num", Pipeline(num_steps), NUM_COLS)]
    if CAT_COLS:
        transformers.append(("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01, sparse_output=False)),
        ]), CAT_COLS))
    return ColumnTransformer(transformers)

MODELS, S_val = {}, {}    # fitted models, validation risk scores (0-100)
def risk(model, X): return model.predict_proba(X)[:, 1] * 100

## Section 1 — Training (one cell per model)

In [ ]:
# 1.0 Rule-based baseline (no learning). Edit RULES after checking the schema.
# (column, condition on that column, points, description) — missing columns are skipped.
RULES = [
    ("failed_attempts_last_hour", lambda s: s >= 3,   30, "3+ failed attempts in last hour"),
    ("is_new_device",             lambda s: s == 1,   20, "new device"),
    ("is_new_country",            lambda s: s == 1,   25, "new country"),
    ("ip_reputation_score",       lambda s: s >= 0.7, 15, "bad IP reputation"),
    ("hour",                      lambda s: s <= 5,   10, "login at night (00-05h)"),
]
ACTIVE_RULES = [r for r in RULES if r[0] in X_all.columns]
skipped = [r[0] for r in RULES if r[0] not in X_all.columns]
if skipped: print("Rules skipped (column not in data):", skipped)
assert ACTIVE_RULES, "No rule matches the schema -> edit RULES"
MAX_POINTS = sum(r[2] for r in ACTIVE_RULES)

def rule_score(X):
    pts = np.zeros(len(X))
    for col, cond, p, _ in ACTIVE_RULES:
        pts += cond(X[col].fillna(0)).astype(int).values * p
    return pts / MAX_POINTS * 100            # 0-100

S_val["Rule baseline"] = rule_score(X_val)
print(f"{len(ACTIVE_RULES)} active rules, max {MAX_POINTS} pts -> scaled to 0-100")

In [ ]:
# 1.1 Logistic Regression (scaled + one-hot)
lr = Pipeline([
    ("prep", make_preprocessor(scale=True)),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE)),
]).fit(X_fit, y_fit)
MODELS["Logistic Regression"] = lr
S_val["Logistic Regression"] = risk(lr, X_val)
print("Logistic Regression trained")

In [ ]:
# 1.2 Random Forest (no scaling needed)
rf = Pipeline([
    ("prep", make_preprocessor(scale=False)),
    ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                   class_weight="balanced_subsample",
                                   n_jobs=-1, random_state=RANDOM_STATE)),
]).fit(X_fit, y_fit)
MODELS["Random Forest"] = rf
S_val["Random Forest"] = risk(rf, X_val)
print("Random Forest trained")

In [ ]:
# 1.3 XGBoost (only if installed)
try:
    from xgboost import XGBClassifier
    pos_w = (y_fit == 0).sum() / max(1, (y_fit == 1).sum())
    xgb = Pipeline([
        ("prep", make_preprocessor(scale=False)),
        ("clf", XGBClassifier(n_estimators=400, max_depth=5, learning_rate=0.05,
                              subsample=0.8, colsample_bytree=0.8,
                              scale_pos_weight=pos_w, eval_metric="aucpr",
                              n_jobs=-1, random_state=RANDOM_STATE)),
    ]).fit(X_fit, y_fit)
    MODELS["XGBoost"] = xgb
    S_val["XGBoost"] = risk(xgb, X_val)
    print("XGBoost trained")
except ImportError:
    print("XGBoost not installed -> skipped (pip install xgboost)")

## Section 2 — Testing (grouped by type of test, one cell per model)

First: shared helpers, test-set risk scores (0–100) and alert thresholds picked on the **validation** slice (never on test).

In [ ]:
# --- Test-set risk scores (0-100) ---
S_test = {"Rule baseline": rule_score(X_test)}
for name, m in MODELS.items():
    S_test[name] = risk(m, X_test)

# --- Alert threshold chosen on VALIDATION: best F1 with false-alarm rate <= FAR_TARGET ---
def far_from_cm(cm):
    tn, fp, fn, tp = cm.ravel()
    return fp / (fp + tn) if (fp + tn) else float("nan")

def pick_threshold(y, s, far_target=FAR_TARGET):
    best_t, best_f1 = 50, -1
    for t in range(1, 100):
        pred = (s >= t).astype(int)
        far = far_from_cm(confusion_matrix(y, pred, labels=[0, 1]))
        f1 = f1_score(y, pred, zero_division=0)
        if far <= far_target and f1 > best_f1:
            best_t, best_f1 = t, f1
    return best_t

THRESH = {n: pick_threshold(y_val, s) for n, s in S_val.items()}
print("alert thresholds (0-100):", THRESH)

RESULTS = {}
def available(name): return name in S_test

# --- test helpers ---
def show_ranking(name):
    if not available(name): return print(f"{name}: not available, skipped")
    s = S_test[name]
    pr, roc = average_precision_score(y_test, s), roc_auc_score(y_test, s)
    RESULTS.setdefault(name, {}).update(pr_auc=pr, roc_auc=roc)
    print(f"{name}: PR-AUC={pr:.4f} (random baseline={y_test.mean():.4f})  ROC-AUC={roc:.4f}")
    fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
    p, r, _ = precision_recall_curve(y_test, s); ax[0].plot(r, p); ax[0].set(title=f"PR - {name}", xlabel="recall", ylabel="precision")
    f, t, _ = roc_curve(y_test, s); ax[1].plot(f, t); ax[1].plot([0, 1], [0, 1], "--"); ax[1].set(title=f"ROC - {name}", xlabel="FPR", ylabel="TPR")
    plt.tight_layout(); plt.show()

def show_threshold(name):
    if not available(name): return print(f"{name}: not available, skipped")
    th = THRESH[name]; pred = (S_test[name] >= th).astype(int)
    cm = confusion_matrix(y_test, pred, labels=[0, 1])
    d = dict(threshold=th, precision=precision_score(y_test, pred, zero_division=0),
             recall=recall_score(y_test, pred, zero_division=0),
             f1=f1_score(y_test, pred, zero_division=0), false_alarm_rate=far_from_cm(cm))
    RESULTS.setdefault(name, {}).update(d)
    print(name, {k: round(v, 4) for k, v in d.items()})

def show_cm(name):
    if not available(name): return print(f"{name}: not available, skipped")
    pred = (S_test[name] >= THRESH[name]).astype(int)
    cm = confusion_matrix(y_test, pred, labels=[0, 1])
    RESULTS.setdefault(name, {})["confusion_matrix"] = cm.tolist()
    ConfusionMatrixDisplay(cm, display_labels=["benign", "attack"]).plot(cmap="Blues")
    plt.title(f"{name} (threshold={THRESH[name]})"); plt.show()
    print(f"false-alarm rate = {far_from_cm(cm):.4f}")

### 2.1 Ranking quality — PR-AUC and ROC-AUC (threshold-free)

In [ ]:
show_ranking("Rule baseline")

In [ ]:
show_ranking("Logistic Regression")

In [ ]:
show_ranking("Random Forest")

In [ ]:
show_ranking("XGBoost")

### 2.2 Alert-level metrics — precision, recall, F1, false-alarm rate (threshold from validation)

In [ ]:
show_threshold("Rule baseline")

In [ ]:
show_threshold("Logistic Regression")

In [ ]:
show_threshold("Random Forest")

In [ ]:
show_threshold("XGBoost")

### 2.3 Confusion matrices

In [ ]:
show_cm("Rule baseline")

In [ ]:
show_cm("Logistic Regression")

In [ ]:
show_cm("Random Forest")

In [ ]:
show_cm("XGBoost")

### 2.4 Explanation (for explaining only, never for selecting features or scores)

In [ ]:
# Rule baseline: how often each rule fires on attacks vs benign logins (test set)
rows = []
for col, cond, p, desc in ACTIVE_RULES:
    fired = cond(X_test[col].fillna(0))
    rows.append(dict(rule=desc, points=p,
                     fires_on_attacks=fired[y_test == 1].mean(),
                     fires_on_benign=fired[y_test == 0].mean()))
display(pd.DataFrame(rows))

In [ ]:
# Logistic Regression: largest standardized coefficients
lr_m = MODELS["Logistic Regression"]
names = lr_m.named_steps["prep"].get_feature_names_out()
coef = pd.Series(lr_m.named_steps["clf"].coef_[0], index=names)
top = coef.reindex(coef.abs().sort_values(ascending=False).index[:15])
top[::-1].plot.barh(figsize=(6, 5), title="Logistic Regression - top coefficients"); plt.tight_layout(); plt.show()

In [ ]:
# Random Forest: SHAP if installed, else impurity importances
def explain_tree(name, n=500):
    m = MODELS.get(name)
    if m is None: return print(f"{name}: not available, skipped")
    Xt = m.named_steps["prep"].transform(X_test.sample(min(n, len(X_test)), random_state=RANDOM_STATE))
    feat = m.named_steps["prep"].get_feature_names_out()
    try:
        import shap
        sv = shap.TreeExplainer(m.named_steps["clf"]).shap_values(Xt)
        if isinstance(sv, list): sv = sv[1]
        elif getattr(sv, "ndim", 2) == 3: sv = sv[:, :, 1]
        shap.summary_plot(sv, Xt, feature_names=feat, show=True)
    except ImportError:
        imp = pd.Series(m.named_steps["clf"].feature_importances_, index=feat).nlargest(15)
        imp[::-1].plot.barh(figsize=(6, 5), title=f"{name} - importances (shap not installed)"); plt.show()
explain_tree("Random Forest")

In [ ]:
explain_tree("XGBoost")

### 2.5 Side-by-side summary and saved outputs

In [ ]:
summary = pd.DataFrame(RESULTS).T.drop(columns="confusion_matrix", errors="ignore")
summary = summary[["pr_auc", "roc_auc", "threshold", "precision", "recall", "f1", "false_alarm_rate"]]
display(summary.sort_values("pr_auc", ascending=False).round(4))

# --- outputs for docs/model_results.md and the demo ---
summary.round(6).to_csv(OUT_DIR / "metrics_summary.csv")
(OUT_DIR / "metrics_full.json").write_text(json.dumps(RESULTS, indent=2))
for n, m in MODELS.items():
    joblib.dump(m, MODEL_DIR / f"{n.lower().replace(' ', '_')}.joblib")

demo = pd.DataFrame({"user_id": df.loc[test_mask, USER_COL].values,
                     "timestamp": df.loc[test_mask, TIME_COL].values,
                     "label": y_test.values})
for n, s in S_test.items():
    demo[f"risk_{n.lower().replace(' ', '_')}"] = np.round(s, 1)
demo.to_csv(OUT_DIR / "demo_risk_scores.csv", index=False)   # ids kept here for display only, never fed to a model
print("saved:", sorted(p.name for p in OUT_DIR.iterdir()), "+ models/")